# Notebook 08 — OCR Image Quality Assessment & Deblur Pipeline
**Memorial Sloan Kettering | Goel Lab**

Evaluates image quality of scanned source documents (PDFs → page images), applies a
deblurring / contrast-enhancement pipeline where needed, and compares OCR text output
before and after preprocessing.

**Sections:**
1. Setup & paths
2. Core functions (blur detection, deblur pipeline, OCR extraction, metrics)
3. Part 1 — Blur detection scan across all source PDFs
4. Part 2 — Deblur pipeline: before vs after OCR comparison
5. Part 3 — Aggregate quality report & visualisations

**Inputs:** `DATA_PRIVATE_DIR/raw/` (source PDFs with PHI) or `DATA_PRIVATE_DIR/deidentified/` (redacted PDFs)

**Outputs (non-PHI, committed):**
- `reports/ocr_blur_scan_results.csv` — per-page blur scores
- `reports/ocr_deblur_comparison.csv` — before/after OCR word counts & precision/recall
- `reports/ocr_blur_score_distribution.png`
- `reports/ocr_deblur_improvement.png`

**Dependencies:** `opencv-python`, `pytesseract`, `Pillow`, `PyMuPDF (fitz)`

Install Tesseract OCR (Windows): https://github.com/UB-Mannheim/tesseract/wiki

Adapted from: `references/ocr_deblur_metrics_prepost.py`

In [ ]:
import os
import re
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import fitz  # PyMuPDF — PDF → image conversion
from PIL import Image
import pytesseract

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(os.getenv('PROJECT_ROOT',
    r'C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca'))
DATA_PRIVATE_DIR = Path(os.getenv('DATA_PRIVATE_DIR', r'C:\Users\jamesr4\loc\data_private'))

# Choose which folder to scan:
#   - raw/        → source PDFs with PHI (run only locally, outputs are non-PHI aggregate stats)
#   - deidentified/pdfs/ → redacted PDFs from NB01
SCAN_DIR   = DATA_PRIVATE_DIR / 'raw'        # change to 'deidentified' / 'pdfs' if preferred
TEXT_DIR   = DATA_PRIVATE_DIR / 'extracted_text'   # ground-truth .txt files from NB04 (optional)
OUTPUT_DIR = PROJECT_ROOT / 'reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tesseract executable path (Windows default)
pytesseract.pytesseract.tesseract_cmd = os.getenv(
    'TESSERACT_CMD', r'C:\Program Files\Tesseract-OCR\tesseract.exe'
)

# ── Config ────────────────────────────────────────────────────────────────────
BLUR_THRESHOLD  = 100   # Laplacian variance below this → blurry
DPI             = 150   # PDF render DPI (150 is fast; 300 for production)
MAX_PAGES       = 3     # Analyse first N pages per PDF (None = all)
TESS_CONFIG     = r'--oem 3 --psm 6 --dpi 300'

print(f'SCAN_DIR  : {SCAN_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'Tesseract : {pytesseract.pytesseract.tesseract_cmd}')

In [ ]:
# ── Core functions ─────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s,.!?;:()\'"\-]', '', text)
    return text.strip()


def detect_blur(image: np.ndarray, threshold: float = BLUR_THRESHOLD) -> tuple:
    """Laplacian variance blur detector. Returns (is_blurry: bool, score: float)."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    return score < threshold, score


def deblur_image(image: np.ndarray) -> np.ndarray:
    """Full preprocessing pipeline: contrast → resize → denoise → threshold → sharpen → morph."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image.copy()

    # Histogram equalisation + contrast stretch
    gray = cv2.equalizeHist(gray)
    gray = cv2.convertScaleAbs(gray, alpha=2.5, beta=50)

    # 2× upscale for finer OCR
    h, w = gray.shape
    gray = cv2.resize(gray, (w * 2, h * 2), interpolation=cv2.INTER_CUBIC)

    # Gaussian denoise
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # Combined Otsu + adaptive threshold
    _, binary_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary_adapt   = cv2.adaptiveThreshold(gray, 255,
                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    binary = cv2.bitwise_and(binary_adapt, binary_otsu)

    # Sharpen
    kernel_sharp = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    binary = cv2.filter2D(binary, -1, kernel_sharp)

    # Morphological cleanup (dilate then erode)
    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.dilate(binary, kernel, iterations=1)
    binary = cv2.erode(binary,  kernel, iterations=1)

    return binary


def extract_text(image: np.ndarray) -> str:
    """Run Tesseract OCR on a numpy image array."""
    pil_img = Image.fromarray(image)
    raw = pytesseract.image_to_string(pil_img, config=TESS_CONFIG)
    return clean_text(raw)


def ocr_metrics(extracted: str, ground_truth: str) -> dict:
    """Word-level precision / recall / F1 against a ground-truth string."""
    ext_words = set(extracted.lower().split())
    gt_words  = set(ground_truth.lower().split())
    tp = len(ext_words & gt_words)
    fp = len(ext_words - gt_words)   # fabricated words
    fn = len(gt_words  - ext_words)  # omitted words
    prec   = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else np.nan
    return {'tp': tp, 'fp': fp, 'fn': fn,
            'precision': prec, 'recall': recall, 'f1': f1}


def pdf_to_images(pdf_path: Path, dpi: int = DPI) -> list:
    """Convert PDF pages to numpy BGR images using PyMuPDF."""
    doc    = fitz.open(str(pdf_path))
    matrix = fitz.Matrix(dpi / 72, dpi / 72)  # 72 dpi is fitz native
    images = []
    for page in doc:
        pix  = page.get_pixmap(matrix=matrix)
        arr  = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        if pix.n == 4:   # RGBA → BGR
            arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2BGR)
        elif pix.n == 3:
            arr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        images.append(arr)
    doc.close()
    return images


print('Core functions defined.')

---
## Part 1: Blur Detection Scan

In [ ]:
# Collect all PDFs in SCAN_DIR recursively
pdf_paths = sorted(SCAN_DIR.rglob('*.pdf'))
print(f'Found {len(pdf_paths)} PDFs in {SCAN_DIR}')

blur_rows = []
for pdf_path in pdf_paths:
    try:
        images = pdf_to_images(pdf_path, dpi=DPI)
        if MAX_PAGES:
            images = images[:MAX_PAGES]
        for page_num, img in enumerate(images, start=1):
            is_blurry, score = detect_blur(img)
            # Relative path keeps it non-PHI (surgeon/case preserved, but that's directory structure)
            rel_path = pdf_path.relative_to(SCAN_DIR)
            blur_rows.append({
                'relative_path': str(rel_path),
                'file_type':     pdf_path.suffix.lower(),
                'page':          page_num,
                'blur_score':    round(score, 2),
                'is_blurry':     is_blurry,
            })
    except Exception as exc:
        print(f'  SKIP {pdf_path.name}: {exc}')

df_blur = pd.DataFrame(blur_rows)
df_blur.to_csv(OUTPUT_DIR / 'ocr_blur_scan_results.csv', index=False)
print(f'\nBlur scan complete: {len(df_blur)} page records')
print(f"  Blurry pages : {df_blur['is_blurry'].sum()} ({df_blur['is_blurry'].mean()*100:.1f}%)")
print(f"  Blur score   : median={df_blur['blur_score'].median():.1f}, "
      f"min={df_blur['blur_score'].min():.1f}, max={df_blur['blur_score'].max():.1f}")
df_blur.head()

In [ ]:
# Visualise blur score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(df_blur[df_blur['is_blurry']]['blur_score'],   bins=40, color='#e74c3c',
        alpha=0.7, label=f'Blurry (< {BLUR_THRESHOLD})', edgecolor='black')
ax.hist(df_blur[~df_blur['is_blurry']]['blur_score'],  bins=40, color='#2ecc71',
        alpha=0.7, label=f'Sharp (≥ {BLUR_THRESHOLD})', edgecolor='black')
ax.axvline(BLUR_THRESHOLD, color='navy', linestyle='--', linewidth=1.5,
           label=f'Threshold = {BLUR_THRESHOLD}')
ax.set_xlabel('Laplacian Variance (blur score)')
ax.set_ylabel('Page count')
ax.set_title('Blur Score Distribution', fontsize=13, fontweight='bold')
ax.legend()

# Box per file type
ax = axes[1]
file_types = df_blur['file_type'].unique()
groups = [df_blur[df_blur['file_type'] == ft]['blur_score'].values for ft in file_types]
ax.boxplot(groups, labels=file_types, patch_artist=True,
           boxprops=dict(facecolor='#95a5a6', alpha=0.7),
           medianprops=dict(color='black', linewidth=2))
ax.axhline(BLUR_THRESHOLD, color='navy', linestyle='--', linewidth=1.2,
           label=f'Threshold = {BLUR_THRESHOLD}')
ax.set_ylabel('Laplacian Variance')
ax.set_title('Blur Score by File Type', fontsize=13, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ocr_blur_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Summary by blurry/sharp
df_blur.groupby('is_blurry')['blur_score'].describe().round(2)

---
## Part 2: Deblur Pipeline — Before vs After OCR

In [ ]:
# Run deblur on blurry pages and compare text output before/after
# Ground-truth text from NB04 extracted_text/ is used if available.

deblur_rows = []
blurry_pdfs = df_blur[df_blur['is_blurry']]['relative_path'].unique()

print(f'Running deblur pipeline on {len(blurry_pdfs)} blurry PDFs...')

for rel_path_str in blurry_pdfs:
    pdf_path = SCAN_DIR / rel_path_str
    # Look for a ground-truth .txt file in TEXT_DIR with same stem
    stem = pdf_path.stem
    gt_candidates = list(TEXT_DIR.rglob(f'{stem}.txt')) if TEXT_DIR.exists() else []
    ground_truth = ''
    if gt_candidates:
        ground_truth = gt_candidates[0].read_text(encoding='utf-8', errors='ignore')

    try:
        images = pdf_to_images(pdf_path, dpi=DPI)
        if MAX_PAGES:
            images = images[:MAX_PAGES]

        for page_num, img in enumerate(images, start=1):
            is_blurry, blur_score = detect_blur(img)
            if not is_blurry:
                continue  # only process pages confirmed blurry

            # OCR before
            text_before = extract_text(img)
            # Deblur + OCR after
            img_deblurred = deblur_image(img)
            text_after = extract_text(img_deblurred)

            row = {
                'relative_path':    rel_path_str,
                'page':             page_num,
                'blur_score':       round(blur_score, 2),
                'words_before':     len(text_before.split()),
                'words_after':      len(text_after.split()),
                'has_ground_truth': bool(ground_truth),
            }

            if ground_truth:
                m_before = ocr_metrics(text_before, ground_truth)
                m_after  = ocr_metrics(text_after,  ground_truth)
                for k, v in m_before.items():
                    row[f'before_{k}'] = round(v, 4) if pd.notna(v) else np.nan
                for k, v in m_after.items():
                    row[f'after_{k}']  = round(v, 4) if pd.notna(v) else np.nan
                row['delta_recall']    = round(m_after['recall']    - m_before['recall'],    4)
                row['delta_precision'] = round(m_after['precision'] - m_before['precision'], 4)
                row['delta_f1']        = round(m_after['f1']        - m_before['f1'],        4)
            else:
                row['delta_words'] = row['words_after'] - row['words_before']

            deblur_rows.append(row)

    except Exception as exc:
        print(f'  SKIP {rel_path_str}: {exc}')

df_deblur = pd.DataFrame(deblur_rows)
df_deblur.to_csv(OUTPUT_DIR / 'ocr_deblur_comparison.csv', index=False)
print(f'\nDeblur comparison: {len(df_deblur)} page records')
df_deblur.head()

---
## Part 3: Aggregate Quality Report

In [ ]:
if df_deblur.empty:
    print('No blurry pages processed — all source documents are sharp.')
else:
    has_gt = df_deblur['has_ground_truth'].any()

    fig, axes = plt.subplots(1, 2 if has_gt else 1, figsize=(14 if has_gt else 7, 5))
    if not has_gt:
        axes = [axes]

    # Word count change
    ax = axes[0]
    if 'delta_words' in df_deblur.columns:
        ax.hist(df_deblur['delta_words'].dropna(), bins=30,
                color='#3498db', edgecolor='black', alpha=0.8)
        ax.axvline(0, color='red', linestyle='--', linewidth=1.2)
        ax.set_xlabel('Δ Word count (after − before)')
        ax.set_ylabel('Page count')
        ax.set_title('Change in OCR Word Count After Deblurring',
                     fontsize=12, fontweight='bold')
        improved = (df_deblur['delta_words'] > 0).sum()
        ax.text(0.97, 0.97, f'{improved}/{len(df_deblur)} pages improved',
                transform=ax.transAxes, ha='right', va='top', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Word count delta not available', transform=ax.transAxes, ha='center')

    # Recall / F1 delta (if ground truth available)
    if has_gt and 'delta_recall' in df_deblur.columns:
        ax = axes[1]
        metrics_delta = df_deblur[['delta_recall', 'delta_precision', 'delta_f1']].dropna()
        ax.boxplot([metrics_delta['delta_recall'].values,
                    metrics_delta['delta_precision'].values,
                    metrics_delta['delta_f1'].values],
                   labels=['Δ Recall', 'Δ Precision', 'Δ F1'],
                   patch_artist=True,
                   boxprops=dict(facecolor='#2ecc71', alpha=0.6),
                   medianprops=dict(color='black', linewidth=2))
        ax.axhline(0, color='red', linestyle='--', linewidth=1.2)
        ax.set_ylabel('Delta (after − before)')
        ax.set_title('OCR Quality Improvement After Deblurring\n(vs ground truth from NB04)',
                     fontsize=12, fontweight='bold')
        ax.grid(axis='y', linestyle=':', alpha=0.5)

    plt.suptitle('OCR Deblur Pipeline: Quality Assessment', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ocr_deblur_improvement.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Summary stats
    print('\nDeblur effect summary:')
    summary_cols = [c for c in ['words_before', 'words_after', 'delta_words',
                                 'before_recall', 'after_recall', 'delta_recall',
                                 'before_f1', 'after_f1', 'delta_f1']
                    if c in df_deblur.columns]
    print(df_deblur[summary_cols].describe().round(3))

In [ ]:
# Final summary table — saved to reports/
summary = pd.DataFrame({
    'Metric': ['Total PDFs scanned', 'Total pages scanned',
               'Blurry pages (n)', 'Blurry pages (%)',
               'Blur score median', 'Blur score min', 'Blur score max',
               'Pages deblurred & OCR compared'],
    'Value': [
        len(pdf_paths),
        len(df_blur),
        int(df_blur['is_blurry'].sum()),
        round(float(df_blur['is_blurry'].mean()) * 100, 1),
        round(float(df_blur['blur_score'].median()), 1),
        round(float(df_blur['blur_score'].min()), 1),
        round(float(df_blur['blur_score'].max()), 1),
        len(df_deblur),
    ]
})
summary.to_csv(OUTPUT_DIR / 'ocr_quality_summary.csv', index=False)
print('Outputs saved to:', OUTPUT_DIR)
summary